## Silver — Transformação

Neste notebook eu levo os dados do Bronze para a Silver. Cada arquivo de origem vira uma tabela Delta no schema `susep_capitalizacao.silver`, limpa, tipada e filtrada para o período 2021–2025.

Os tratamentos aplicados aqui foram decididos no perfil de qualidade do Bronze (`02_bronze_perfil_qualidade`).

### 1. Leitura do Bronze

A Silver é construída a partir do Bronze, que guarda os 4 CSVs originais da SUSEP no Volume, sem nenhuma alteração. Como são arquivos, e não tabelas, o primeiro passo é carregá-los no Spark para poder transformá-los.

Leio cada arquivo com o separador `;` e o seu encoding. Todas as colunas entram como texto, porque os valores usam vírgula decimal (`84,91`); a conversão para número é feita no SQL das próximas etapas. Cada arquivo vira uma temp view com o nome do próprio arquivo, o que permite consultá-lo com SQL.

In [0]:
BASE = "/Volumes/susep_capitalizacao/bronze/raw_susep_capitalizacao"

def ler(arquivo, encoding):
    # sem inferSchema: tudo entra como texto; a tipagem é feita no SQL
    return spark.read.csv(f"{BASE}/{arquivo}", header=True, sep=";", encoding=encoding)

bronze = {
    "ses_cap_uf":    ler("ses_cap_uf.csv",    "ASCII"),
    "Ses_Dados_Cap": ler("Ses_Dados_Cap.csv", "ISO-8859-1"),
    "Ses_prov":      ler("Ses_prov.csv",      "ASCII"),
    "Ses_cias":      ler("Ses_cias.csv",      "windows-1252"),
}

for nome, df in bronze.items():
    df.createOrReplaceTempView(nome)
    print(f"{nome:<14} {df.count():>7,} linhas")

### 2. Schema Silver

Crio o schema `silver` no catálogo `susep_capitalizacao`, que vai receber as tabelas tratadas. O `IF NOT EXISTS` permite rodar o notebook de novo sem erro.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS susep_capitalizacao.silver

### 3. Criação da tabela `silver.empresas`

Crio a tabela com o cadastro das companhias de capitalização, que dá origem à `dim_empresa` na Gold. O `Ses_cias` cobre todos os mercados da SUSEP (769 empresas), então mantenho só as que aparecem nos arquivos de capitalização (`ses_cap_uf` e `Ses_Dados_Cap`) no período 2021–2025.

Tratamentos: `trim` em `Coenti` e `Noenti`, que vêm com espaços nas pontas. `Cogrupo` e `Nogrupo` ficam de fora, porque estão vazias. Resultado: 19 empresas.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.silver.empresas AS
WITH empresas_cap AS (          -- empresas com movimento de capitalização no período
  SELECT TRIM(COENTI) AS coenti FROM ses_cap_uf
  WHERE DAMESANO BETWEEN '202101' AND '202512'
  UNION                         -- UNION (sem ALL) já remove repetidos
  SELECT TRIM(coenti) FROM Ses_Dados_Cap
  WHERE damesano BETWEEN '202101' AND '202512'
)
SELECT
  TRIM(c.Coenti) AS coenti,
  TRIM(c.Noenti) AS nome_empresa
FROM Ses_cias c
JOIN empresas_cap e ON TRIM(c.Coenti) = e.coenti;

SELECT COUNT(*) AS linhas, COUNT(DISTINCT coenti) AS empresas
FROM susep_capitalizacao.silver.empresas;

### 4. Criação da tabela `silver.cap_uf`

Crio a tabela de prêmios, resgates e sorteios por empresa, mês e UF, filtrada para 2021–2025, com as colunas renomeadas conforme o catálogo. Valores em R$ viram `decimal(18,2)` (a vírgula decimal é trocada por ponto antes da conversão) e as quantidades viram inteiro.

Tratamentos (do perfil de qualidade):
- `upper` em `UF` (há um caso `Am`).
- Negativos → 0 em `RESGPAGO`, `SORTPAGO`, `RESGATANTES` e `SORTEIOS`.
- `RESGATANTES` e `SORTEIOS` arredondados para inteiro.
- `PREMIO` negativo é mantido: é valor líquido de devoluções e cancelamentos.
- `NUMPARTIC` fica de fora do modelo.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.silver.cap_uf AS
SELECT
  TRIM(COENTI)                                                  AS coenti,
  CAST(DAMESANO AS INT)                                         AS damesano,
  UPPER(TRIM(UF))                                               AS uf,
  CAST(REPLACE(PREMIO, ',', '.') AS DECIMAL(18,2))              AS premio,
  GREATEST(CAST(REPLACE(RESGPAGO, ',', '.') AS DECIMAL(18,2)), 0) AS resgate_pago,
  GREATEST(CAST(REPLACE(SORTPAGO, ',', '.') AS DECIMAL(18,2)), 0) AS sorteio_pago,
  GREATEST(CAST(ROUND(CAST(REPLACE(RESGATANTES, ',', '.') AS DOUBLE)) AS BIGINT), 0) AS qtd_resgatantes,
  GREATEST(CAST(ROUND(CAST(REPLACE(SORTEIOS,    ',', '.') AS DOUBLE)) AS BIGINT), 0) AS qtd_sorteios
FROM ses_cap_uf
WHERE DAMESANO BETWEEN '202101' AND '202512';

SELECT COUNT(*) AS linhas, COUNT(DISTINCT uf) AS ufs, COUNT(DISTINCT coenti) AS empresas
FROM susep_capitalizacao.silver.cap_uf;

### 5. Criação da tabela `silver.cap_modalidade`

Crio a tabela de receitas, resgates e sorteios pagos por empresa, mês e modalidade, filtrada para 2021–2025, com as colunas renomeadas conforme o catálogo. Valores em R$ viram `decimal(18,2)`.

Tratamentos (do perfil de qualidade):
- Modalidade de código 0 vem sem descrição → "Não informada".
- `receitasCap` negativo é mantido: é valor líquido de devoluções e cancelamentos, como o `PREMIO`.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.silver.cap_modalidade AS
SELECT
  TRIM(coenti)                                                  AS coenti,
  CAST(damesano AS INT)                                         AS damesano,
  CAST(codModal AS INT)                                         AS cod_modalidade,
  COALESCE(NULLIF(TRIM(modalidade), ''), 'Não informada')       AS modalidade,
  CAST(REPLACE(receitasCap,   ',', '.') AS DECIMAL(18,2))       AS receitas,
  CAST(REPLACE(valorResg,     ',', '.') AS DECIMAL(18,2))       AS resgates,
  CAST(REPLACE(sorteiosPagos, ',', '.') AS DECIMAL(18,2))       AS sorteios_pagos
FROM Ses_Dados_Cap
WHERE damesano BETWEEN '202101' AND '202512';

SELECT COUNT(*) AS linhas, COUNT(DISTINCT cod_modalidade) AS modalidades, COUNT(DISTINCT coenti) AS empresas
FROM susep_capitalizacao.silver.cap_modalidade;

### 6. Criação da tabela `silver.provisao`

Crio a tabela com o saldo mensal da provisão total por empresa, filtrada para 2021–2025, com as colunas renomeadas conforme o catálogo. O `Ses_prov` cobre todos os mercados supervisionados pela SUSEP, então mantenho só as empresas da `silver.empresas`. O valor em R$ vira `decimal(18,2)`.

Tratamento (do perfil de qualidade): remoção de duplicatas exatas com `DISTINCT`. As 5 encontradas no perfil são de 01/2001 e já ficam fora pelo filtro de período, mas a regra fica no pipeline para proteger cargas futuras.

In [0]:
%sql
CREATE OR REPLACE TABLE susep_capitalizacao.silver.provisao AS
SELECT DISTINCT
  TRIM(p.coenti)                                     AS coenti,
  CAST(p.damesano AS INT)                            AS damesano,
  CAST(REPLACE(p.valor, ',', '.') AS DECIMAL(18,2))  AS provisao_total
FROM Ses_prov p
JOIN susep_capitalizacao.silver.empresas e ON TRIM(p.coenti) = e.coenti
WHERE p.damesano BETWEEN '202101' AND '202512';

SELECT COUNT(*) AS linhas, COUNT(DISTINCT coenti) AS empresas
FROM susep_capitalizacao.silver.provisao;

### 7. Checagem de completude

Concilio a quantidade de linhas do Bronze com a da Silver. No Bronze, aplico as mesmas regras de filtro (período 2021–2025 e empresas de capitalização) recalculadas a partir dos próprios CSVs. A diferença deve ser zero: nenhuma linha se perdeu ou duplicou na transformação.

In [0]:
%sql
WITH empresas_cap AS (          -- mesma regra da silver.empresas, calculada no Bronze
  SELECT TRIM(COENTI) AS coenti FROM ses_cap_uf    WHERE DAMESANO BETWEEN '202101' AND '202512'
  UNION
  SELECT TRIM(coenti)           FROM Ses_Dados_Cap WHERE damesano BETWEEN '202101' AND '202512'
),
contagens AS (
  SELECT 'empresas' AS tabela,
         (SELECT COUNT(*) FROM Ses_cias)                                                      AS bronze_total,
         (SELECT COUNT(*) FROM Ses_cias WHERE TRIM(Coenti) IN (SELECT coenti FROM empresas_cap)) AS bronze_filtrado,
         (SELECT COUNT(*) FROM susep_capitalizacao.silver.empresas)                           AS silver
  UNION ALL
  SELECT 'cap_uf',
         (SELECT COUNT(*) FROM ses_cap_uf),
         (SELECT COUNT(*) FROM ses_cap_uf WHERE DAMESANO BETWEEN '202101' AND '202512'),
         (SELECT COUNT(*) FROM susep_capitalizacao.silver.cap_uf)
  UNION ALL
  SELECT 'cap_modalidade',
         (SELECT COUNT(*) FROM Ses_Dados_Cap),
         (SELECT COUNT(*) FROM Ses_Dados_Cap WHERE damesano BETWEEN '202101' AND '202512'),
         (SELECT COUNT(*) FROM susep_capitalizacao.silver.cap_modalidade)
  UNION ALL
  SELECT 'provisao',
         (SELECT COUNT(*) FROM Ses_prov),
         (SELECT COUNT(*) FROM Ses_prov WHERE damesano BETWEEN '202101' AND '202512'
                                          AND TRIM(coenti) IN (SELECT coenti FROM empresas_cap)),
         (SELECT COUNT(*) FROM susep_capitalizacao.silver.provisao)
)
SELECT *, silver - bronze_filtrado AS diferenca
FROM contagens